# GLM-4.7-Flash STEM REAP Pruning (Official Quality)

**REAP** (Router-weighted Expert Activation Pruning) for MoE models.

## Key Implementation Details

Based on [CerebrasResearch/reap](https://github.com/CerebrasResearch/reap) official implementation:

**REAP Formula:**
```
S_j = (1/|X_j|) × Σ_{x∈X_j} g_j(x) · ||f_j(x)||_2
```

Where:
- `X_j` = tokens where expert j was selected (in top-k)
- `g_j(x)` = router weight for expert j on token x  
- `f_j(x)` = output of expert j on token x
- `||.||_2` = L2 norm

**This notebook:**
- **Base model:** zai-org/GLM-4.7-Flash (64 experts)
- **Target:** 43 experts (33% pruning)
- **Calibration:** Siesher/mits-calibration-dataset (1,490 bilingual STEM examples)
- **Output:** GGUF for LMStudio/llama.cpp

**Requirements:** Colab Pro+ with A100 40GB

## 1. Проверка GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {gpu_mem:.1f} GB")
    if gpu_mem < 35:
        print(f"WARNING: A100 40GB recommended!")
else:
    raise RuntimeError("No CUDA GPU!")

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
PyTorch: 2.9.0+cu126
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.5 GB


## 2. Установка зависимостей

In [2]:
# Установка зависимостей
# transformers >= 5.0 нужен для glm4_moe_lite

!pip uninstall -y tensorflow tensorflow-cpu tf-keras -q 2>/dev/null || true
!pip install -q --upgrade pip
!pip install -q datasets huggingface_hub accelerate sentencepiece tqdm

# Transformers из git (с поддержкой glm4_moe_lite)
!pip install -q git+https://github.com/huggingface/transformers.git

import transformers
print(f"[OK] transformers: {transformers.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.6 MB/s eta 0:00:0000:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[OK] transformers: 5.0.1.dev0


In [3]:
# Создаём директории
!mkdir -p /content/models /content/outputs /content/gguf

print("[OK] Directories created")

[OK] Directories created


In [4]:
# Hugging Face login (для доступа к модели)
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
# Скачиваем модель GLM-4.7-Flash
from huggingface_hub import snapshot_download
from transformers import AutoConfig
import os

MODEL_ID = "zai-org/GLM-4.7-Flash"
MODEL_PATH = "/content/models/glm-4.7-flash"

if not os.path.exists(f"{MODEL_PATH}/config.json"):
    print(f"Downloading {MODEL_ID}...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_PATH,
        ignore_patterns=["*.gguf", "*.md", "*.txt"]
    )

# Проверяем конфигурацию
config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
print(f"\n[OK] Model: {config.model_type}")
print(f"Experts: {config.n_routed_experts}")
print(f"Layers: {config.num_hidden_layers}")
print(f"Active per token: {config.num_experts_per_tok}")

Fetching 57 files:   0%|          | 0/57 [00:00<?, ?it/s]


[OK] Model: glm4_moe_lite
Experts: 64
Layers: 47
Active per token: 4


## 3. REAP Pruning

Official-quality implementation based on [CerebrasResearch/reap](https://github.com/CerebrasResearch/reap).

In [6]:
!rm -rf /content/outputs/glm-stem-pruned

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from datasets import load_dataset
from tqdm import tqdm
import gc
import json
from safetensors.torch import save_file, load_file
import glob
import re

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
MODEL_PATH = "/content/models/glm-4.7-flash"
DATASET_ID = "Siesher/mits-calibration-dataset"
OFFLOAD_FOLDER = "/content/offload"
COMPRESSION_RATIO = 0.33
N_CALIBRATION_SAMPLES = 500

os.makedirs(OFFLOAD_FOLDER, exist_ok=True)


class REAPObserverOptimized:
    def __init__(self, model, num_layers: int, num_experts: int, num_experts_per_tok: int):
        self.model = model
        self.num_layers = num_layers
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.device = next(model.parameters()).device

        self.reap_sum = torch.zeros(num_layers, num_experts, device=self.device)
        self.reap_count = torch.zeros(num_layers, num_experts, device=self.device)
        self.hooks = []
        self.total_tokens = 0

    def _compute_expert_output_norm(self, hidden, experts, expert_idx):
        gate_up = F.linear(hidden, experts.gate_up_proj[expert_idx])
        mid = gate_up.shape[-1] // 2
        activated = F.silu(gate_up[..., :mid]) * gate_up[..., mid:]
        output = F.linear(activated, experts.down_proj[expert_idx])
        return torch.linalg.norm(output, dim=-1)

    def _create_hook(self, layer_idx: int):
        def hook(module, args, output):
            hidden_states = args[0]
            if hidden_states.dim() == 2:
                hidden_states = hidden_states.unsqueeze(0)

            batch_size, seq_len, hidden_size = hidden_states.shape
            num_tokens = batch_size * seq_len
            self.total_tokens += num_tokens

            router_logits = module.gate(hidden_states)
            routing_weights = F.softmax(router_logits, dim=-1, dtype=torch.float32)
            topk_weights, topk_indices = torch.topk(routing_weights, self.num_experts_per_tok, dim=-1)

            hidden_flat = hidden_states.view(num_tokens, hidden_size)
            topk_indices_flat = topk_indices.view(num_tokens, self.num_experts_per_tok)
            routing_flat = routing_weights.view(num_tokens, self.num_experts)
            experts = module.experts

            with torch.no_grad():
                unique_experts = topk_indices_flat.unique()
                for expert_idx in unique_experts.tolist():
                    active_mask = (topk_indices_flat == expert_idx).any(dim=-1)
                    if not active_mask.any():
                        continue

                    active_hidden = hidden_flat[active_mask]
                    active_weights = routing_flat[active_mask, expert_idx]
                    expert_norms = self._compute_expert_output_norm(active_hidden, experts, expert_idx)

                    self.reap_sum[layer_idx, expert_idx] += (active_weights * expert_norms).sum()
                    self.reap_count[layer_idx, expert_idx] += active_mask.sum()
        return hook

    def register_hooks(self):
        for layer_idx in range(self.num_layers):
            layer = self.model.model.layers[layer_idx]
            if hasattr(layer.mlp, 'experts'):
                hook = layer.mlp.register_forward_hook(self._create_hook(layer_idx))
                self.hooks.append(hook)
        print(f"Registered {len(self.hooks)} hooks")

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks = []

    def get_global_saliency(self):
        layer_means = self.reap_sum / self.reap_count.clamp(min=1)
        valid_layers = (self.reap_count.sum(dim=1) > 0)
        return layer_means[valid_layers].mean(dim=0) if valid_layers.sum() > 0 else layer_means.mean(dim=0)

    def get_experts_to_keep(self, target_experts: int):
        saliency = self.get_global_saliency()
        _, top_indices = torch.topk(saliency, target_experts)
        return sorted(top_indices.cpu().tolist())


def prune_and_save_from_files(model_path, experts_to_keep, target_experts, output_dir):
    """Prune experts stored as separate keys (experts.0, experts.1, etc.)"""

    # Build mapping: old expert index -> new expert index
    experts_to_keep_sorted = sorted(experts_to_keep)
    old_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(experts_to_keep_sorted)}
    experts_set = set(experts_to_keep_sorted)

    print(f"Keeping experts: {experts_to_keep_sorted}")
    print(f"Mapping: old_idx -> new_idx")

    os.makedirs(output_dir, exist_ok=True)

    # Update and save config
    config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
    config.n_routed_experts = target_experts
    config.save_pretrained(output_dir)

    # Copy tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tokenizer.save_pretrained(output_dir)

    # Pattern to match expert keys: model.layers.X.mlp.experts.N.weight_name
    expert_pattern = re.compile(r'(model\.layers\.\d+\.mlp\.experts\.)(\d+)(\..*)')

    safetensor_files = sorted(glob.glob(os.path.join(model_path, "*.safetensors")))
    print(f"Found {len(safetensor_files)} weight files")

    pruned_state_dict = {}

    for sf_path in tqdm(safetensor_files, desc="Processing weight files"):
        weights = load_file(sf_path)

        for key, value in weights.items():
            match = expert_pattern.match(key)

            if match:
                # This is an expert weight
                prefix = match.group(1)  # model.layers.X.mlp.experts.
                expert_idx = int(match.group(2))  # N
                suffix = match.group(3)  # .down_proj.weight etc

                if expert_idx in experts_set:
                    # Keep this expert with new index
                    new_idx = old_to_new[expert_idx]
                    new_key = f"{prefix}{new_idx}{suffix}"
                    pruned_state_dict[new_key] = value
                # else: skip this expert

            elif '.mlp.gate.weight' in key:
                # Router weights: select rows for kept experts
                pruned_state_dict[key] = value[experts_to_keep_sorted].clone()

            elif '.mlp.gate.e_score_correction_bias' in key:
                # Router bias: select for kept experts
                pruned_state_dict[key] = value[experts_to_keep_sorted].clone()

            else:
                # Keep as-is (embeddings, attention, etc.)
                pruned_state_dict[key] = value

        del weights
        gc.collect()

    # Save with sharding
    print(f"Saving {len(pruned_state_dict)} tensors...")
    total_size = sum(t.numel() * t.element_size() for t in pruned_state_dict.values())
    max_shard_size = 5 * 1024 * 1024 * 1024

    if total_size <= max_shard_size:
        save_file(pruned_state_dict, os.path.join(output_dir, "model.safetensors"))
    else:
        current_shard, current_size, shard_idx = {}, 0, 1
        index = {"weight_map": {}, "metadata": {"total_size": total_size}}

        for key, tensor in tqdm(pruned_state_dict.items(), desc="Sharding"):
            tensor_size = tensor.numel() * tensor.element_size()
            if current_size + tensor_size > max_shard_size and current_shard:
                shard_name = f"model-{shard_idx:05d}-of-XXXXX.safetensors"
                save_file(current_shard, os.path.join(output_dir, shard_name))
                shard_idx += 1
                current_shard, current_size = {}, 0
            current_shard[key] = tensor
            current_size += tensor_size
            index["weight_map"][key] = f"model-{shard_idx:05d}-of-XXXXX.safetensors"

        if current_shard:
            save_file(current_shard, os.path.join(output_dir, f"model-{shard_idx:05d}-of-XXXXX.safetensors"))

        total_shards = shard_idx
        for key in index["weight_map"]:
            index["weight_map"][key] = index["weight_map"][key].replace("XXXXX", f"{total_shards:05d}")

        for i in range(1, total_shards + 1):
            old = os.path.join(output_dir, f"model-{i:05d}-of-XXXXX.safetensors")
            new = os.path.join(output_dir, f"model-{i:05d}-of-{total_shards:05d}.safetensors")
            if os.path.exists(old):
                os.rename(old, new)

        with open(os.path.join(output_dir, "model.safetensors.index.json"), "w") as f:
            json.dump(index, f, indent=2)

    print(f"[OK] Saved to {output_dir}")


def run_reap_pruning():
    print("="*60)
    print("GLM-4.7-Flash REAP Pruning")
    print("="*60)

    config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
    num_experts = config.n_routed_experts
    num_layers = config.num_hidden_layers
    num_experts_per_tok = config.num_experts_per_tok
    target_experts = int(num_experts * (1 - COMPRESSION_RATIO))

    print(f"Experts: {num_experts} -> {target_experts} ({COMPRESSION_RATIO*100:.0f}% reduction)")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("\nLoading model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        offload_folder=OFFLOAD_FOLDER
    )
    model.eval()

    dataset = load_dataset(DATASET_ID, split="train")
    print(f"Dataset: {len(dataset)} samples")

    observer = REAPObserverOptimized(model, num_layers, num_experts, num_experts_per_tok)
    observer.register_hooks()

    print(f"\nCalibrating on {N_CALIBRATION_SAMPLES} samples...")
    with torch.no_grad():
        for i in tqdm(range(min(N_CALIBRATION_SAMPLES, len(dataset)))):
            text = dataset[i]['instruction']
            if dataset[i].get('output'):
                text += "\n" + dataset[i]['output'][:500]
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            try:
                model(**{k: v.to(model.device) for k, v in inputs.items()})
            except:
                continue

    observer.remove_hooks()
    print(f"Tokens processed: {observer.total_tokens:,}")

    experts_to_keep = observer.get_experts_to_keep(target_experts)
    saliency = observer.get_global_saliency()
    print(f"\nSaliency: min={saliency.min():.4f}, max={saliency.max():.4f}")

    # Free memory
    del model, observer
    gc.collect()
    torch.cuda.empty_cache()

    print("\nPruning and saving...")
    prune_and_save_from_files(MODEL_PATH, experts_to_keep, target_experts, OUTPUT_DIR)

    # Save metadata
    metadata = {
        "base_model": "zai-org/GLM-4.7-Flash",
        "method": "REAP",
        "calibration_dataset": DATASET_ID,
        "calibration_samples": N_CALIBRATION_SAMPLES,
        "original_experts": num_experts,
        "pruned_experts": target_experts,
        "compression_ratio": COMPRESSION_RATIO,
        "experts_kept": experts_to_keep,
        "saliency_scores": saliency.cpu().tolist()
    }
    with open(f"{OUTPUT_DIR}/reap_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print("\n" + "="*60)
    print("REAP Pruning Complete!")
    print("="*60)

    return experts_to_keep

In [5]:
!rm -rf /content/outputs/glm-stem-pruned

In [6]:
experts_kept = run_reap_pruning()

GLM-4.7-Flash REAP Pruning
Experts: 64 -> 42 (33% reduction)

Loading model...


Loading weights:   0%|          | 0/751 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 768.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 434.88 MiB is free. Process 252162 has 39.12 GiB memory in use. Of the allocated memory 37.33 GiB is allocated by PyTorch, and 1.39 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [12]:
# Полная очистка памяти
import gc         
import torch
                                                                                                                                                                                                                                        
# Удаляем все переменные модели
for name in list(globals().keys()):
    if 'model' in name.lower() or 'observer' in name.lower():
        try:
            del globals()[name]
        except:
            pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Проверяем
print(f"Free GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB used")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

Free GPU memory: 12.38 GB used
memory.used [MiB], memory.free [MiB]
38131 MiB, 2374 MiB


## 4. Верификация

In [10]:
# Верификация pruned модели
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"

config = AutoConfig.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
print(f"Experts after pruning: {config.n_routed_experts}")

model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

# Проверяем структуру
l1 = model.model.layers[1].mlp
print(f"layer1.n_routed_experts: {l1.n_routed_experts}")
print(f"gate_up_proj.shape[0]: {l1.experts.gate_up_proj.shape[0]}")
print(f"gate.weight.shape[0]: {l1.gate.weight.shape[0]}")
print("[OK] Model verified")

Experts after pruning: 42


Loading weights:   0%|          | 0/751 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 504.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 244.88 MiB is free. Process 5286 has 39.31 GiB memory in use. Of the allocated memory 37.51 GiB is allocated by PyTorch, and 1.31 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import IPython                                                                                                                                   
IPython.Application.instance().kernel.do_shutdown(restart=True)

{'status': 'ok', 'restart': True}

: 

In [2]:
# Верификация pruned модели
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig         
import torch
import os                                                                                                                                                                                                                               

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
OFFLOAD_FOLDER = "/content/offload_verify"
os.makedirs(OFFLOAD_FOLDER, exist_ok=True)

config = AutoConfig.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
print(f"Experts after pruning: {config.n_routed_experts}")

model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    offload_folder=OFFLOAD_FOLDER
)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

# Проверяем структуру
l1 = model.model.layers[1].mlp
print(f"layer1.n_routed_experts: {l1.n_routed_experts}")
print(f"gate_up_proj.shape[0]: {l1.experts.gate_up_proj.shape[0]}")
print(f"gate.weight.shape[0]: {l1.gate.weight.shape[0]}")
print("[OK] Model structure verified")

# Тест генерации
prompts = ["Solve: 2x + 5 = 13", "Напиши функцию для проверки числа на простоту"]
model.eval()
for p in prompts:
    print(f"\n> {p}")
    inp = tokenizer(p, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(out[0], skip_special_tokens=True)[len(p):].strip()[:200])

print("\n[OK] Generation works!")

`torch_dtype` is deprecated! Use `dtype` instead!


Experts after pruning: 42


Loading weights:   0%|          | 0/751 [00:00<?, ?it/s]

Glm4MoeLiteForCausalLM LOAD REPORT from: /content/outputs/glm-stem-pruned
Key                                                 | Status     |                                                                                                     
----------------------------------------------------+------------+-----------------------------------------------------------------------------------------------------
model.layers.47.post_attention_layernorm.weight     | UNEXPECTED |                                                                                                     
model.layers.47.mlp.experts.down_proj               | UNEXPECTED |                                                                                                     
model.layers.47.self_attn.q_a_layernorm.weight      | UNEXPECTED |                                                                                                     
model.layers.47.self_attn.kv_a_layernorm.weight     | UNEXPECTED |                    

RuntimeError: You set `ignore_mismatched_sizes` to `False`, thus raising an error. For details look at the above report!

In [3]:
# Проверка сохранённых весов напрямую
from safetensors.torch import load_file      
import glob
import os                                                                                                                                                                                                                               

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"

# Найти все safetensors файлы
files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.safetensors")))
print(f"Found {len(files)} safetensors files")

# Загрузить первый файл и проверить размеры expert весов
for f in files[:1]:
    print(f"\nChecking: {os.path.basename(f)}")
    weights = load_file(f)

    for key in sorted(weights.keys()):
        if 'layers.1.mlp' in key:
            print(f"  {key}: {weights[key].shape}")
    break

# Проверить config
from transformers import AutoConfig
config = AutoConfig.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
print(f"\nConfig n_routed_experts: {config.n_routed_experts}")

Found 12 safetensors files

Checking: model-00001-of-00012.safetensors
  model.layers.1.mlp.experts.0.down_proj.weight: torch.Size([2048, 1536])
  model.layers.1.mlp.experts.0.gate_proj.weight: torch.Size([1536, 2048])
  model.layers.1.mlp.experts.0.up_proj.weight: torch.Size([1536, 2048])
  model.layers.1.mlp.experts.1.down_proj.weight: torch.Size([2048, 1536])
  model.layers.1.mlp.experts.1.gate_proj.weight: torch.Size([1536, 2048])
  model.layers.1.mlp.experts.1.up_proj.weight: torch.Size([1536, 2048])
  model.layers.1.mlp.experts.10.down_proj.weight: torch.Size([2048, 1536])
  model.layers.1.mlp.experts.10.gate_proj.weight: torch.Size([1536, 2048])
  model.layers.1.mlp.experts.10.up_proj.weight: torch.Size([1536, 2048])
  model.layers.1.mlp.experts.11.down_proj.weight: torch.Size([2048, 1536])
  model.layers.1.mlp.experts.11.gate_proj.weight: torch.Size([1536, 2048])
  model.layers.1.mlp.experts.11.up_proj.weight: torch.Size([1536, 2048])
  model.layers.1.mlp.experts.12.down_proj.w

In [ ]:
# Тест генерации
prompts = [
    "Solve: 2x + 5 = 13",
    "Напиши код на Python для проверки числа на простоту",
    "Explain Newton's second law"
]

model.eval()
for p in prompts:
    print(f"\n> {p}")
    inp = tokenizer(p, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(p):].strip()
    print(response[:300])

In [ ]:
# Очистка памяти перед GGUF конвертацией
del model
import gc; gc.collect()
torch.cuda.empty_cache()
print("[OK] Memory cleared")

## 5. GGUF Конвертация

In [ ]:
# Установка llama.cpp
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp 2>/dev/null || true
!pip install -q gguf numpy
print("[OK] llama.cpp ready")

In [ ]:
# Конвертация в GGUF FP16
import os
OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
GGUF_FP16 = "/content/gguf/glm-stem-pruned-f16.gguf"

%cd /content/llama.cpp
!python convert_hf_to_gguf.py "{OUTPUT_DIR}" --outfile "{GGUF_FP16}" --outtype f16

if os.path.exists(GGUF_FP16):
    print(f"[OK] FP16: {os.path.getsize(GGUF_FP16)/1e9:.2f} GB")

In [ ]:
# Компиляция квантайзера
%cd /content/llama.cpp
!make llama-quantize -j$(nproc) 2>/dev/null || echo "Already compiled"

In [ ]:
# Квантование в Q4_K_M и Q8_0
import os
GGUF_FP16 = "/content/gguf/glm-stem-pruned-f16.gguf"
GGUF_Q4 = "/content/gguf/glm-stem-pruned-q4km.gguf"
GGUF_Q8 = "/content/gguf/glm-stem-pruned-q8.gguf"

%cd /content/llama.cpp

if os.path.exists(GGUF_FP16):
    !./llama-quantize "{GGUF_FP16}" "{GGUF_Q4}" Q4_K_M
    !./llama-quantize "{GGUF_FP16}" "{GGUF_Q8}" Q8_0
    
    print("\nGGUF files:")
    for f in [GGUF_Q4, GGUF_Q8]:
        if os.path.exists(f):
            print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1e9:.2f} GB")
    
    # Удаляем FP16 для экономии места
    os.remove(GGUF_FP16)
    print("\n[OK] Quantized files ready")

## 6. Upload на HuggingFace (опционально)

In [ ]:
# Upload на HuggingFace (установи UPLOAD=True)
UPLOAD = False
REPO = "Siesher/glm-stem-pruned-gguf"

if UPLOAD:
    from huggingface_hub import upload_file, create_repo
    create_repo(REPO, exist_ok=True)
    
    for f in ["/content/gguf/glm-stem-pruned-q4km.gguf", "/content/gguf/glm-stem-pruned-q8.gguf"]:
        if os.path.exists(f):
            print(f"Uploading {os.path.basename(f)}...")
            upload_file(f, os.path.basename(f), REPO)
    print(f"[OK] https://huggingface.co/{REPO}")
else:
    print("[SKIP] Set UPLOAD=True to upload to HuggingFace")

## 7. Summary

In [ ]:
# Summary
import os
import json

print("="*60)
print("GLM-4.7-Flash STEM REAP Pruning - COMPLETE")
print("="*60)

# Metadata
try:
    with open("/content/outputs/glm-stem-pruned/reap_metadata.json") as f:
        m = json.load(f)
    print(f"\nMethod: {m['method']}")
    print(f"Experts: {m['original_experts']} -> {m['pruned_experts']}")
    print(f"Calibration: {m['calibration_dataset']}")
except:
    pass

# Files
print("\nOutput files:")
files = [
    "/content/outputs/glm-stem-pruned",
    "/content/gguf/glm-stem-pruned-q4km.gguf",
    "/content/gguf/glm-stem-pruned-q8.gguf"
]
for f in files:
    if os.path.exists(f):
        if ".gguf" in f:
            print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1e9:.2f} GB")
        else:
            print(f"  HF model: {f}")

print("\n" + "="*60)
print("Usage:")
print("  - LMStudio: Load .gguf file directly")
print("  - Ollama: ollama create model -f Modelfile")
print("  - llama.cpp: ./llama-cli -m model.gguf -p 'prompt'")
print("="*60)